In [0]:
%run ../../02_common_utils/operations

In [0]:
from pyspark.sql.functions import *

In [0]:
team_name="team_lemma"
catalog_name=f"charles_schwab_retailbrokerage_dev_{team_name}"
staging_schema=f"staging"
dbutils.widgets.text("batch_id","1","BATCH ID")
bronze_prospect=f"{catalog_name}.bronze.prospect"
staging_prospect=f"{catalog_name}.{staging_schema}.prospect_current"
silver_prospect=f"{catalog_name}.silver.prospect"


In [0]:
batch_id=dbutils.widgets.get("batch_id")

In [0]:
df_bronze=spark.read.table(bronze_prospect)
df_bronze.createOrReplaceTempView("df_bronze")
# df_bronze.count()

In [0]:
# df_bronze.printSchema()

In [0]:
#Type Casting
df_bronze=df_bronze.withColumn("Age",col("Age").cast("bigint"))\
    .withColumn("Income",col("Income").cast("decimal(15,2)"))\
    .withColumn("NetWorth",col("NetWorth").cast("decimal(15,2)"))\
    .withColumn("CreditRating",col("CreditRating").cast("int"))\
    .withColumn("NumberCreditCards",col('NumberCreditCards').cast("int"))\
    .withColumn("NumberCars",col("NumberCars").cast("int"))\
    .withColumn("NumberChildren",col("NumberChildren").cast("int"))\
       

In [0]:
#hash for all 22 column excluding metadata cols
cols_to_hash=[c for c in df_bronze.columns if c not in ("_batch","_run_id","_source_file","_ingest_ts")]
# print(len(cols_to_hash))
df_hashed=df_bronze.withColumn("row_hash",md5(concat_ws("||",*[col(c) for c in cols_to_hash])))
df_hashed.createOrReplaceTempView("df_hashed")
# df_hashed.printSchema()
# df_hashed.count()

In [0]:
#first batch id in full history
df_first_batch=df_hashed.groupBy("AgencyID","row_hash")\
    .agg(min(col("_batch")).cast("int").alias("first_batchid"))
# df_first_batch.limit(10).display()

In [0]:
#current data with widget batch id
df_current=df_hashed.filter(col("_batch")==batch_id)

In [0]:
# Staging
df_staging=df_current.join(df_first_batch,["AgencyID","row_hash"],"left")

In [0]:
# df_silver = spark.read.table(silver_prospect).filter(col("is_active") == True)
# df_silver.limit(10).display()

In [0]:
#for batch 2 and batch 3 table will already present
if spark.catalog.tableExists(staging_prospect):
    df_silver = spark.read.table(silver_prospect).filter(col("is_active") == True)
    df_silver_hash = df_silver.select(col("AgencyID"), col("row_hash").alias("silver_hash"))
    
    df_staging = df_staging.join(df_silver_hash, ["AgencyID"], "left")
    
    df_staging = df_staging.withColumn(
        "CDC_Action",
        when(col("silver_hash").isNull(), lit("N"))
        .when(col("row_hash") == col("silver_hash"), lit("X"))
        .otherwise(lit("C"))
    ).drop("silver_hash")
else:
    df_staging=df_staging.withColumn("CDC_Action",lit("N"))

In [0]:
df_final=df_staging.withColumn("_load_ts",current_timestamp())

In [0]:
try:
    #Write with overwrite because this is full refresh process
    print(f"Writing to table {staging_prospect}")
    df_final.write.format("delta").mode("overwrite").saveAsTable(staging_prospect)
    print(f"Write completed....")
    run_id=df_final.select("_run_id").first()[0]

    history_df=spark.sql(f"DESCRIBE HISTORY {staging_prospect}").first()
    source_count=int(history_df["operationMetrics"]["numOutputRows"])
    target_count=df_staging.count()
    source_count = target_count
    log_pipeline_recon(
                    spark=spark,
        run_id=run_id,
        batch_id=batch_id,
        domain="CUSTOMER",
        table_name="prospect_current",
        source_layer="bronze",
        target_layer="staging",
        source_count=source_count,
        target_count=target_count
    )
    log_audit_event(
        spark=spark,
        run_id=run_id,
        batch=batch_id,
        layer="staging",
        table_name="prospect_current",
        operation="OVERWRITE",
        rows_affected=target_count
    )
except Exception as e:
    print(f"Error: {e}")

In [0]:
spark.table(staging_prospect).count()